<a href="https://colab.research.google.com/github/duongngusmart/NhapMonPhanTichDuLieuHocSau/blob/main/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile patient_heart_rate.csv
1.0,Micky Mous,56.0,70kgs,72,69,71,,,
2.0,Daisy Duck,34.0,154lbs,,,,74,73,75
3.0,Scrooge McDuck,45.0,78kgs,78,79,72,,,
4.0,Pink Panther,54.0,90kgs,,,,69,,75
5.0,Huey McDuck,52.0,85kgs,,,,68,75,72
6.0,Dewey McDuck,19.0,56kgs,,,,71,78,75
7.0,Scööpy Doo,32.0,78kgs,78,76,75,,,
1.0,Micky Mous,56.0,70kgs,72,69,71,,,
,,,,,,,,,
8.0,Minnie Mous,,65kgs,,,,74,72,73
9.0,Goofy Dog,40.0,,75,76,77,,,
10.0,Pluto Dog,,,70,71,72,,,

Writing patient_heart_rate.csv


In [2]:
import pandas as pd
# Problem 1: Thêm header và đọc dữ liệu
column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df = pd.read_csv("patient_heart_rate.csv", names=column_names)

In [3]:
# Problem 2: Tách cột Name thành Firstname và Lastname
df[['Firstname', 'Lastname']] = df['Name'].str.split(expand=True)
df = df.drop('Name', axis=1)

In [4]:
# Problem 3: Đồng nhất đơn vị cột Weight (lbs sang kgs)
weight = df['Weight'].astype(str)
for i in range(len(weight)):
    x = weight.iloc[i]
    if "lbs" in x[-3:]:
        x = x[:-3]
        y = int(float(x) / 2.2)
        weight.iloc[i] = str(y) + "kgs"
df['Weight'] = weight

In [5]:
# Problem 4: Xóa các dòng rỗng hoàn toàn
df.dropna(how="all", inplace=True)

In [6]:
# Problem 5: Xóa dòng trùng lặp
df = df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'])

In [8]:
# Problem 6: Loại bỏ ký tự non-ASCII trong tên
df['Firstname'] = df['Firstname'].replace(r'[^\x00-\x7F]+', '', regex=True)
df['Lastname'] = df['Lastname'].replace(r'[^\x00-\x7F]+', '', regex=True)

In [11]:
# Problem 7: Xử lý dữ liệu thiếu (Age, Weight)

# 1. Xóa dòng nếu thiếu cả Age và Weight (gán lại vào df thay vì inplace=True)
df = df.dropna(subset=['Age', 'Weight'], how='all')

# 2. Điền giá trị trung bình (Mean) cho cột Age bị thiếu
df['Age'] = df['Age'].fillna(df['Age'].mean())

# 3. Điền phần Weight bị thiếu bằng giá trị trung bình
# Do Weight đang ở dạng chuỗi (ví dụ '70kgs'), ta cần tách phần số ra để tính toán
df['Weight_num'] = df['Weight'].str.extract(r'(\d+)').astype(float)
df['Weight_num'] = df['Weight_num'].fillna(df['Weight_num'].mean())

# Ghép lại chữ 'kgs' và xóa cột tạm
df['Weight'] = df['Weight_num'].astype(int).astype(str) + 'kgs'
df = df.drop('Weight_num', axis=1)

In [12]:
# Problem 8: Phân rã dữ liệu từ cột ngang thành dòng (Melt)
df_melted = pd.melt(df, id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'],
                    value_name="PulseRate", var_name="sex_and_time").sort_values(['Id', 'Age'])

# Tách thông tin giới tính và thời gian từ cột 'sex_and_time'
tmp_df = df_melted["sex_and_time"].str.extract(r'(\D)(\d+)(\d{2})', expand=True)
tmp_df.columns = ["Sex", "hours_lower", "hours_upper"]
tmp_df["Time"] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]

# Nối dữ liệu đã tách và xóa các cột trung gian thừa
df_clean = pd.concat([df_melted, tmp_df], axis=1)
df_clean = df_clean.drop(['sex_and_time', 'hours_lower', 'hours_upper'], axis=1)

# Xóa các dòng rác sinh ra trong quá trình melt nếu Id bị NaN
df_clean = df_clean.dropna(subset=['Id'])

# --- Xử lý dữ liệu thiếu cho PulseRate (Huyết áp) ---
df_clean['PulseRate'] = pd.to_numeric(df_clean['PulseRate'], errors='coerce')

# Điền giá trị huyết áp bằng trung bình liền trước và sau (interpolate) theo từng bệnh nhân
df_clean['PulseRate'] = df_clean.groupby(['Id', 'Firstname'])['PulseRate'].transform(
    lambda group: group.interpolate(method='linear', limit_direction='both')
)

# Nếu vẫn còn sót NaN, điền bằng trung bình toàn bộ dữ liệu (gán lại biến, không dùng inplace)
df_clean['PulseRate'] = df_clean['PulseRate'].fillna(df_clean['PulseRate'].mean())

# Rút gọn dữ liệu, reset lại index và lưu file
df_clean = df_clean.reset_index(drop=True)
df_clean.to_csv('patient_heart_rate_clean.csv', index=False)
print("Đã làm sạch và lưu file thành công (không còn cảnh báo)!")

Đã làm sạch và lưu file thành công (không còn cảnh báo)!
